# Lab 03 — ゾーンとコードのマスターを整備

OneLake ショートカット `Files/taxi_zone` の CSV（共有 ADLS Gen2 上に残ったまま）を読み込み、
ゾーンマスター `taxi_zone_lookup` と、ベンダー・料金コード・支払方法・乗車種別のコード表を Delta に保存します。

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

ZONE_FILE = "Files/taxi_zone/Taxi_zone_lookup.csv"
zones = (
    spark.read.option("header", "true").csv(ZONE_FILE)
    .select(F.col("LocationID").cast("int").alias("LocationID"), F.trim("Borough").alias("Borough"),
            F.trim("Zone").alias("Zone"), F.trim("service_zone").alias("service_zone"))
)
zones.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi_zone_lookup")

schema = StructType([StructField("code", IntegerType(), False), StructField("name", StringType(), False)])
LOOKUPS = {
    "vendor_lookup": ("VendorID", "vendor_name", [(1, "Creative Mobile Technologies, LLC"), (2, "VeriFone Inc.")]),
    "ratecode_lookup": ("RatecodeID", "ratecode_name", [(1, "Standard rate"), (2, "JFK"), (3, "Newark"),
                        (4, "Nassau or Westchester"), (5, "Negotiated fare"), (6, "Group ride")]),
    "payment_type_lookup": ("payment_type", "payment_type_name", [(1, "Credit card"), (2, "Cash"), (3, "No charge"),
                            (4, "Dispute"), (5, "Unknown"), (6, "Voided trip")]),
    "trip_type_lookup": ("trip_type", "trip_type_name", [(1, "Street-hail"), (2, "Dispatch")]),
}
saved = {"taxi_zone_lookup": spark.table("taxi_zone_lookup").count()}
for table, (key, label, rows) in LOOKUPS.items():
    (spark.createDataFrame(rows, schema).withColumnRenamed("code", key).withColumnRenamed("name", label)
     .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table))
    saved[table] = spark.table(table).count()

save_result("nb_03_prepare_dimensions", {"source": ZONE_FILE, "savedTables": saved,
                                          "next": "答え合わせは nb_03_validate_dimensions で実行します。"})